# Chapter 1: Transformer models

In [1]:
# Setup: install deps and authenticate to the Hub. Re-run on every fresh runtime.
%pip install -q transformers datasets evaluate accelerate  # add peft when needed

import os
from getpass import getpass
from huggingface_hub import whoami
try:
    from google.colab import userdata  # works only in the Colab web UI
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN_COURSE")
except Exception as e:
    print("Colab secret not read:", repr(e))
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Paste your HF token (hidden, not saved): ")
HF_USER = whoami()["name"]  # HF libraries read HF_TOKEN from the env; this checks it works
print("Logged in to the Hub as", HF_USER)

Colab secret not read: TimeoutException('Requesting secret HF_TOKEN_COURSE timed out. Secrets can only be fetched when running from the Colab UI.')
Logged in to the Hub as pedropalvareez


In [2]:
# Check the GPU: expect a Tesla T4 with ~15 GB of memory.
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Pipeline demos

In [4]:
# Demo 1: sentiment analysis. pipeline() = tokenizer + model + post-processing in one call.
from transformers import pipeline

classifier = pipeline("sentiment-analysis")  # no model named: downloads a default English classifier
classifier([
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
        "Great, another Monday.",
    "The food was fine but the service was terrible.",

])

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455},
 {'label': 'POSITIVE', 'score': 0.9997097849845886},
 {'label': 'NEGATIVE', 'score': 0.9920613765716553}]

In [5]:
# Demo 2: zero-shot classification. Labels are chosen at call time; no training on them.
classifier = pipeline("zero-shot-classification")  # default: an NLI model (~1.6 GB download)
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445985913276672, 0.11197450757026672, 0.04342687129974365]}

In [7]:
# Demo 3: text generation. The model predicts the next token, appends it, repeats.
# Model named explicitly: the default (SmolLM3-3B, ~12 GB in float32) is too big for a CPU runtime.
generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M")
generator(
    "In this course, we will teach you how to",
    max_new_tokens=30,        # how many tokens to add after the prompt
    num_return_sequences=2,   # sample two different continuations
)

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

[{'generated_text': 'In this course, we will teach you how to use this powerful tool to create stunning and engaging video content. We will also explore some of the latest trends and technologies in video production to help you stand'},
 {'generated_text': "In this course, we will teach you how to play with and create the magic of the Internet. We'll provide you with the tools and knowledge needed to understand and make the most out of the various"}]

In [8]:
# Demo 4: mask filling. An encoder predicts the hidden word using context on BOTH sides.
unmasker = pipeline("fill-mask", model="distilbert/distilroberta-base")
unmasker("This course will teach you all about <mask> models.", top_k=2)  # <mask> is this model's mask token

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[{'score': 0.19619713723659515,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.040527280420064926,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

In [10]:
# Demo 5: named entity recognition (NER). One label per token, then tokens merged into entities.
ner = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple",  # merge word pieces ("Hu", "##gging") into whole entities
)
ner("My name is Sylvain, spanish and I work at Hugging Face in Brooklyn.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.9979986),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'MISC',
  'score': np.float32(0.666492),
  'word': '##ish',
  'start': 24,
  'end': 27},
 {'entity_group': 'ORG',
  'score': np.float32(0.9774205),
  'word': 'Hugging Face',
  'start': 42,
  'end': 54},
 {'entity_group': 'LOC',
  'score': np.float32(0.9929802),
  'word': 'Brooklyn',
  'start': 58,
  'end': 66}]

In [14]:
# Demos 6-8, v5 style: transformers v5 removed the question-answering, summarization and translation
# pipelines. One instruction-tuned chat model does all three; the task is described in the prompt.
chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")  # multilingual, ~2 GB in float32

def ask(prompt):
    messages = [{"role": "user", "content": prompt}]           # chat format: a list of role/content messages
    out = chat(messages, max_new_tokens=80, do_sample=False)   # greedy: same answer every run
    return out[0]["generated_text"][-1]["content"]             # last message = the model's reply

text = """America has changed dramatically during recent years. Not only has the number of graduates
in traditional engineering disciplines such as mechanical, civil, electrical, chemical, and aeronautical
engineering declined, but in most of the premier American universities engineering curricula now
concentrate on and encourage largely the study of engineering science. As a result, there are declining
offerings in engineering subjects dealing with infrastructure, the environment, and related issues,
and greater concentration on high technology subjects, largely supporting increasingly complex
scientific developments. While the latter is important, it should not be at the expense of more
traditional engineering."""

print("QA:", ask("Context: My name is Sylvain and I work at Hugging Face in Brooklyn. "
                 "Question: Where do I work? Answer in a few words."))
print("Summary:", ask("Summarize in one sentence: " + text))
print("Translation:", ask("Translate to English: Este curso está producido por Hugging Face."))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

QA: I am currently employed at Hugging Face in Brooklyn.
Summary: The decline in engineering education offerings at prestigious American universities reflects a shift towards focusing on advanced scientific fields like high technology rather than traditional engineering disciplines.
Translation: The course is produced by Hugging Face.


In [15]:
# Demo 9: bias. A pretrained model reproduces the statistics of its training text, stereotypes included.
unmasker = pipeline("fill-mask", model="google-bert/bert-base-uncased")  # BERT uses [MASK], not <mask>
for sentence in ["This man works as a [MASK].", "This woman works as a [MASK]."]:
    print(sentence, [r["token_str"] for r in unmasker(sentence)])  # top 5 guesses

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

This man works as a [MASK]. ['carpenter', 'lawyer', 'farmer', 'businessman', 'doctor']
This woman works as a [MASK]. ['nurse', 'maid', 'teacher', 'waitress', 'prostitute']
